In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git


In [ ]:
import torch
import clip
import numpy as np
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:
model, preprocess = clip.load("ViT-B/32", device=device)
print("CLIP model loaded")


In [ ]:
dataset = load_dataset("garythung/trashnet")
print(dataset)


In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


In [ ]:
def collate_fn(batch):
    images = [preprocess(item["image"]) for item in batch]
    labels = [item["label"] for item in batch]
    images = torch.stack(images)
    return images, labels

loader = DataLoader(
    dataset["train"],
    batch_size=64,        # good for P100
    shuffle=False,
    collate_fn=collate_fn
)

X, y = [], []

model.eval()
with torch.no_grad():
    for images, labels in tqdm(loader):
        images = images.to(device)
        emb = model.encode_image(images)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        X.append(emb.cpu().numpy())
        y.extend(labels)

X = np.vstack(X)
y = np.array(y)

print("Embeddings created:", X.shape, y.shape)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=2000)
clf.fit(X, y)

preds = clf.predict(X)
acc = accuracy_score(y, preds)

print("Linear Probe Accuracy:", acc)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

test_preds = clf.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)

print("Test Accuracy:", test_acc)


Zero-shot CLIP

In [6]:
!pip install -q git+https://github.com/openai/CLIP.git


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00


In [ ]:
import torch
import clip
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Load CLIP
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()


In [ ]:
# Class names
class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# Text prompts (zero-shot)
text_prompts = [f"a photo of {c} waste" for c in class_names]

# Encode text
with torch.no_grad():
    text_tokens = clip.tokenize(text_prompts).to(device)
    text_features = model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)


In [7]:
from datasets import load_dataset

dataset = load_dataset("garythung/trashnet")
print(dataset)


README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

dataset-original.zip:   0%|          | 0.00/3.63G [00:00<?, ?B/s]

dataset-resized.zip:   0%|          | 0.00/42.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5054 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5054
    })
})


In [ ]:
y_true = []
y_pred = []

for item in tqdm(dataset["train"]):
    image = item["image"]
    label = item["label"]

    # Preprocess image
    image_input = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_input)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # Cosine similarity
        similarity = (image_features @ text_features.T).squeeze(0)
        pred = similarity.argmax().item()

    y_true.append(label)
    y_pred.append(pred)


In [ ]:
acc = accuracy_score(y_true, y_pred)
print("Zero-Shot CLIP Accuracy:", acc)


In [ ]:
# Waste classes
class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# Prompt templates (keep generic, not dataset-specific)
prompt_templates = [
    "a photo of {} waste",
    "a photo of {} trash",
    "a discarded {} item",
    "a piece of {} garbage",
    "a photo of {} material waste"
]


In [ ]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

# Build prompts for each class
all_class_text_features = []

with torch.no_grad():
    for cls in class_names:
        prompts = [template.format(cls) for template in prompt_templates]
        tokens = clip.tokenize(prompts).to(device)
        text_features = model.encode_text(tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Average prompts for this class
        class_feature = text_features.mean(dim=0)
        class_feature /= class_feature.norm()
        
        all_class_text_features.append(class_feature)

# Stack to shape: (num_classes, feature_dim)
text_features = torch.stack(all_class_text_features)


In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

y_true, y_pred = [], []

for item in tqdm(dataset["train"]):
    image = item["image"]
    label = item["label"]

    image_input = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_input)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # Cosine similarity with ensemble text features
        similarity = (image_features @ text_features.T).squeeze(0)
        pred = similarity.argmax().item()

    y_true.append(label)
    y_pred.append(pred)


In [ ]:
acc = accuracy_score(y_true, y_pred)
print("Zero-Shot CLIP (Prompt Ensemble) Accuracy:", acc)

print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
text_features = []

with torch.no_grad():
    for cls in material_classes:
        prompts = material_prompts[cls]
        text_tokens = clip.tokenize(prompts).to(device)
        embeddings = model.encode_text(text_tokens)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        text_features.append(embeddings.mean(dim=0))

text_features = torch.stack(text_features)  # shape: (5, 512)


In [ ]:
class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in enumerate(class_names)}

print(id2label)


In [ ]:
material_classes = ["cardboard", "glass", "metal", "paper", "plastic"]

all_classes = material_classes + ["trash"]

id2label = {
    0: "cardboard",
    1: "glass",
    2: "metal",
    3: "paper",
    4: "plastic",
    5: "trash"
}

print("Classes loaded:", material_classes)


In [ ]:
prompts = [f"a photo of {c} waste" for c in material_classes]

with torch.no_grad():
    text_tokens = clip.tokenize(prompts).to(device)
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("Text features shape:", text_features.shape)


In [ ]:
threshold = 0.22   # starting value

y_true = []
y_pred = []

for item in tqdm(dataset["train"]):
    image = item["image"]
    true_label = id2label[item["label"]]

    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        sims = (image_features @ text_features.T).squeeze(0).cpu().numpy()

    max_score = sims.max()
    idx = sims.argmax()

    if max_score < threshold:
        pred_label = "trash"
    else:
        pred_label = material_classes[idx]

    y_true.append(true_label)
    y_pred.append(pred_label)


In [ ]:
print("Hierarchical Zero-Shot Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))


prompt improvement


In [ ]:
material_classes = ["cardboard", "glass", "metal", "paper", "plastic"]


In [ ]:
templates = [
    "a photo of {c} waste",
    "a close-up photo of {c} trash",
    "a photo of discarded {c}",
    "a photo of thrown away {c} waste",
    "a photo of recyclable {c} material",
    "a photo of household {c} garbage"
]


In [ ]:
text_features = []

with torch.no_grad():
    for c in material_classes:
        prompts = [template.format(c=c) for template in templates]
        tokens = clip.tokenize(prompts).to(device)
        embeddings = model.encode_text(tokens)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        
        class_embedding = embeddings.mean(dim=0)
        class_embedding = class_embedding / class_embedding.norm()
        
        text_features.append(class_embedding)

text_features = torch.stack(text_features)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_true = []
y_pred = []

for item in tqdm(dataset["train"]):
    image = item["image"]
    true_label = id2label[item["label"]]

    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        sims = (image_features @ text_features.T).squeeze(0)

        pred_idx = sims.argmax().item()
        pred_label = material_classes[pred_idx]

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Prompt-Ensemble Zero-Shot Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

thresholds = [0.15, 0.18, 0.20, 0.22, 0.25, 0.28, 0.30]

results = []

for threshold in thresholds:
    y_true = []
    y_pred = []

    for item in tqdm(dataset["train"]):
        image = item["image"]
        true_label = id2label[item["label"]]

        image_tensor = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            image_features = model.encode_image(image_tensor)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            similarities = (image_features @ text_features.T).squeeze(0).cpu().numpy()

        max_score = similarities.max()
        idx = similarities.argmax()

        if max_score < threshold:
            pred_label = "trash"
        else:
            pred_label = material_classes[idx]

        y_true.append(true_label)
        y_pred.append(pred_label)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")

    results.append((threshold, acc, f1))
    print(f"Threshold {threshold:.2f} → Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")

print("\nFINAL RESULTS:")
for r in results:
    print(r)


In [3]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)


CUDA available: True
Using device: cuda


In [4]:
print(torch.cuda.get_device_name(0))


Tesla T4


In [5]:
import torch
import clip
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, f1_score


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()


CUDA available: True
Using device: cuda


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [9]:
# Get class names directly from dataset
labels = dataset["train"].features["label"].names

# Create mappings
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in id2label.items()}

# Separate material classes (remove trash)
material_classes = labels.copy()
if "trash" in material_classes:
    material_classes.remove("trash")

print("All labels:", labels)
print("Material classes:", material_classes)
print("id2label:", id2label)


All labels: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Material classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic']
id2label: {0: 'cardboard', 1: 'glass', 2: 'metal', 3: 'paper', 4: 'plastic', 5: 'trash'}


In [12]:
def zero_shot_batch_eval(dataset, batch_size=64):
    y_true = []
    y_pred = []

    prompts = [f"a photo of {c} waste" for c in material_classes]

    with torch.no_grad():
        text_tokens = clip.tokenize(prompts).to(device)
        text_features = model.encode_text(text_tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)

    for i in tqdm(range(0, len(dataset["train"]), batch_size)):
        batch = dataset["train"][i:i+batch_size]

        images = batch["image"]
        labels_batch = [id2label[l] for l in batch["label"]]

        image_tensors = torch.stack([preprocess(img) for img in images]).to(device)

        with torch.no_grad():
            image_features = model.encode_image(image_tensors)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            sims = image_features @ text_features.T

        preds = sims.argmax(dim=1).cpu().numpy()

        y_true.extend(labels_batch)
        y_pred.extend([material_classes[p] for p in preds])

    return y_true, y_pred


In [13]:
y_true, y_pred = zero_shot_batch_eval(dataset)

print("Zero-shot Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))


100%|██████████| 79/79 [10:19<00:00,  7.84s/it]

Zero-shot Accuracy: 0.6721408785120696
              precision    recall  f1-score   support

   cardboard       0.58      0.86      0.69       806
       glass       0.60      0.89      0.72      1002
       metal       0.86      0.53      0.66       820
       paper       0.68      0.77      0.72      1188
     plastic       0.87      0.48      0.62       964
       trash       0.00      0.00      0.00       274

    accuracy                           0.67      5054
   macro avg       0.60      0.59      0.57      5054
weighted avg       0.68      0.67      0.65      5054




/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
from sklearn.metrics import accuracy_score, classification_report, f1_score
from tqdm import tqdm

threshold = 0.25   # starting value

y_true = []
y_pred = []

# Build prompt ensemble text features
templates = [
    "a photo of {} waste",
    "a photo of discarded {}",
    "a photo of recyclable {} material",
    "a close-up photo of {} trash",
    "an image of {} garbage"
]

with torch.no_grad():
    text_features = []
    for c in material_classes:
        prompts = [t.format(c) for t in templates]
        tokens = clip.tokenize(prompts).to(device)
        emb = model.encode_text(tokens)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        class_emb = emb.mean(dim=0)
        class_emb = class_emb / class_emb.norm()
        text_features.append(class_emb)
    text_features = torch.stack(text_features)

# Inference
for item in tqdm(dataset["train"]):
    image = item["image"]
    true_label = id2label[item["label"]]

    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        sims = (image_features @ text_features.T).squeeze(0)

    max_score = sims.max().item()
    idx = sims.argmax().item()

    if max_score < threshold:
        pred_label = "trash"
    else:
        pred_label = material_classes[idx]

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Hierarchical Prompt-Ensemble Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred))


100%|██████████| 5054/5054 [10:54<00:00,  7.73it/s]

Hierarchical Prompt-Ensemble Accuracy: 0.5609418282548476
Macro F1: 0.5490572932513341
              precision    recall  f1-score   support

   cardboard       0.85      0.84      0.85       806
       glass       0.58      0.70      0.63      1002
       metal       0.86      0.47      0.61       820
       paper       0.65      0.52      0.58      1188
     plastic       0.87      0.42      0.57       964
       trash       0.04      0.17      0.06       274

    accuracy                           0.56      5054
   macro avg       0.64      0.52      0.55      5054
weighted avg       0.71      0.56      0.61      5054



cross dataset testing

In [2]:
!pip install -q kagglehub


In [3]:
import kagglehub

path = kagglehub.dataset_download("mostafaabla/garbage-classification")

print("Dataset path:", path)


Dataset path: /kaggle/input/garbage-classification


In [4]:
import os

for root, dirs, files in os.walk(path):
    print(root, dirs)
    break


/kaggle/input/garbage-classification ['garbage_classification']


In [7]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

print("Using device:", device)


100%|████████████████████████████████████████| 338M/338M [00:00<00:00, 368MiB/s]


Using device: cuda


In [8]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

dataset_b = ImageFolder(root=path, transform=preprocess)
loader_b = DataLoader(dataset_b, batch_size=64, shuffle=False)

print("Total images:", len(dataset_b))
print("Class mapping:", dataset_b.class_to_idx)


Total images: 15515
Class mapping: {'garbage_classification': 0}


In [9]:
import os

# Go one level deeper
correct_path = os.path.join(path, "garbage_classification")

print("Correct dataset path:", correct_path)

# Reload dataset
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

dataset_b = ImageFolder(root=correct_path, transform=preprocess)
loader_b = DataLoader(dataset_b, batch_size=64, shuffle=False)

print("Total images:", len(dataset_b))
print("Class mapping:", dataset_b.class_to_idx)


Correct dataset path: /kaggle/input/garbage-classification/garbage_classification
Total images: 15515
Class mapping: {'battery': 0, 'biological': 1, 'brown-glass': 2, 'cardboard': 3, 'clothes': 4, 'green-glass': 5, 'metal': 6, 'paper': 7, 'plastic': 8, 'shoes': 9, 'trash': 10, 'white-glass': 11}


In [10]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import torch

class_names = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

# Build text embeddings
prompts = [f"a photo of {c} waste" for c in class_names]

with torch.no_grad():
    text_tokens = clip.tokenize(prompts).to(device)
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

y_true = []
y_pred = []

print(f"Total batches: {len(loader_b)}")

# Loop with live progress bar
for images, labels in tqdm(loader_b, total=len(loader_b), desc="Cross-dataset inference"):
    images = images.to(device)
    labels = labels.cpu().numpy()

    with torch.no_grad():
        image_features = model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        sims = image_features @ text_features.T

    preds = sims.argmax(dim=1).cpu().numpy()

    y_true.extend(labels)
    y_pred.extend(preds)

# Final metrics
print("\nCross-Dataset Zero-Shot Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


Total batches: 243


Cross-dataset inference: 100%|██████████| 243/243 [02:18<00:00,  1.75it/s]


Cross-Dataset Zero-Shot Accuracy: 0.06670963583628746


ValueError: Number of classes, 12, does not match size of target_names, 6. Try specifying the labels parameter

In [1]:
print(sorted([v for v in globals().keys()
              if ("y" in v.lower()) or ("pred" in v.lower())]))


['get_ipython']


In [2]:
y_true.csv
y_pred.csv


NameError: name 'y_true' is not defined